In [ ]:
import math
import numpy as np
import copy

FRAMES = 20
POPULATION_SIZE = 10
MAX_SPEED = 10

#### Hidden Layer
Theres 3 neurons on this hidden layer. 
For each neuron (Z), we multiply all inputs (x) by 4 weights (W) and a bias (B):  
$$Z_n = (x1 * w_n1) + (x2 * w_n2) + (x3 * w_n3) + (x4 * w_n4) + b_n$$
After that we need to normalize/activate the neuron's value (A) to be betwen -1 and 1. For that we use tanh:
$$ A_n = tanh(Z_n) \iff \frac{(e^z - e^{-z})}{(e^z + e^{-z})} $$
The output of the neuron will be wathever the tanh() is.

In [991]:
class NeuralNetwork:
    def __init__(self, p_pos, f_pos):
        self.player_pos = p_pos
        self.fruit_pos = f_pos
        self.layer_1_weights = self.generate_random_weights(3, 4)
        self.layer_1_bias = self.generate_random_biases(3)
        self.layer_2_weights = self.generate_random_weights(2, 3)
        self.layer_2_bias = self.generate_random_biases(2)

    def generate_random_weights(self, rows, cols):
        return np.random.uniform(-1, 1, (rows, cols))  # generate a matrix of random weights between -1 and 1

    def generate_random_biases(self, size):
        return np.random.uniform(-1, 1, size)  # generate a list of random bias between -1 and 1

    def forward(self):

        # convert the position of the player and the fruit into a list
        raw_values = [self.player_pos[0], self.player_pos[1], self.fruit_pos[0], self.fruit_pos[1]]

        # LAYER 1
        Z_layer_1 = []
        for neuron_weights in self.layer_1_weights:  # for each row of weights
            z_in_row = sum(n * w for n, w in zip(raw_values, neuron_weights))  # multiply and sum the values and weights
            Z_layer_1.append(z_in_row)

        Z_layer_1 = [v + b for v, b in zip(Z_layer_1, self.layer_1_bias)]  # add bias

        A_layer_1 = [math.tanh(v) for v in Z_layer_1]  # normalize

        # LAYER 2
        Z_layer_2 = []
        for neuron_weights in self.layer_2_weights:  # for each row of weights
            Z_out_row = sum(n * w for n, w in zip(A_layer_1, neuron_weights))  # multiply and sum the values and weights
            Z_layer_2.append(Z_out_row)

        Z_layer_2 = [v + b for v, b in zip(Z_layer_2, self.layer_2_bias)]  # add bias

        A_layer_2 = [math.tanh(v) for v in Z_layer_2]  # normalize

        # OUTPUT
        angle = (A_layer_2[0] + 1) / 2 * 360  # convert normilizaed value to angle
        speed = (A_layer_2[1] + 1) / 2 * MAX_SPEED  # convert normalized value to speed

        # get the postion of the player based on angle and speed
        angle_rad = math.radians(angle)
        self.player_pos = (
            self.player_pos[0] + math.cos(angle_rad) * speed,
            self.player_pos[1] + math.sin(angle_rad) * speed
        )

        return {"angle": angle, "speed": speed}

    def mutate(self, rate=0.1):
        # add a small random noise to the weights and biases
        self.layer_1_weights += np.random.uniform(-rate, rate, self.layer_1_weights.shape)
        self.layer_1_bias += np.random.uniform(-rate, rate, self.layer_1_bias.shape)
        self.layer_2_weights += np.random.uniform(-rate, rate, self.layer_2_weights.shape)
        self.layer_2_bias += np.random.uniform(-rate, rate, self.layer_2_bias.shape)

In [992]:
class AllNetworks:
    def __init__(self, player_pos, fruit_pos):
        self.player_position = player_pos
        self.fruit_position = fruit_pos
        # create an intial list of neural networks with fully random weights
        self.neural_networks = [NeuralNetwork(self.player_position[:], self.fruit_position[:]) for _ in range(POPULATION_SIZE)]
        self.best_ai = None

    def forward(self):
        for n in self.neural_networks:  # move all networks by one tick
            n.forward()

    def getBest(self):
        closest = FRAMES * MAX_SPEED * 1.1

        for n in self.neural_networks:
            dx = n.player_pos[0] - n.fruit_pos[0]
            dy = n.player_pos[1] - n.fruit_pos[1]
            distance = math.sqrt(dx**2 + dy**2)
            if distance < closest:
                closest = distance
                self.best_ai = n  # set the best ai as the one that got the closest to the fruit

        print(self.best_ai.player_pos, closest)
        self.best_ai.player_pos = self.player_position[:]   # reset the position of the neural network

    def reproduce(self):
        # create a new list of neural networks with the weighs of the best ai
        self.neural_networks = [copy.deepcopy(self.best_ai) for _ in range(POPULATION_SIZE)]

    def mutate(self):
        for n in self.neural_networks[1:]:  # mutade all networks by adding a small random noise, except the first one
            n.mutate(rate=0.01)

In [ ]:
networks = AllNetworks((1, 1), (100, 100))

In [994]:

for n in range(FRAMES):
    networks.forward()

networks.getBest()
networks.reproduce()
networks.mutate()

(58.82530376416062, 43.87943683650996) 69.60512351760407
